# Lesion Gate: "Is there a skin lesion in this photo?" (step 1 of the Pi pipeline)

The Pi runs two models on each photo:
1. **This gate** (MobileNetV3-Small, yes/no): is there a skin lesion at all? If not, the Pi says
   "no skin lesion found" and stops.
2. **The lesion-type model** from `lesion_type_pipeline.ipynb` (EfficientNet-B0, 7 types): only runs
   when the gate says yes.

Without the gate, the lesion model always picks one of its 7 types, even for a photo of a wall.

The gate is **purely binary**: one output, the probability that the photo shows a skin lesion. The
assumption is that the user photographs a spot on their own skin with the camera, so the main "no"
case is healthy skin. Non-skin photos are extra "no" examples in case the camera points at
something else. The model never tells the two "no" kinds apart; they only appear separately in the
results.

What this notebook does:
1. Builds a yes/no dataset. **Yes** = HAM10000 dermoscopy + PAD-UFES-20 phone photos. **No** =
   healthy skin cut from the corners of those photos, plus everyday non-skin photos (COCO val2017).
2. Splits it **by lesion** (a lesion's photo and its healthy-skin corner stay in the same split).
3. Fine-tunes MobileNetV3-Small with the same quality degradation as the lesion model, so blurry
   photos of real lesions aren't rejected.
4. Picks the **threshold** that lets about 99% of real lesions through (missing a lesion is worse
   than a false pass), and **calibrates** the confidence shown to the patient.
5. Measures accuracy, sensitivity, specificity, precision, F1 and AUC at each quality drop.
6. Exports FP32 and INT8 ONNX and runs the full two-step pipeline on example photos.

**This is a research prototype, not a medical device.** The confidence is a triage signal, never a
diagnosis. Run with the working directory set to `notebooks/`.

In [ ]:
import os, io, json, random, warnings, copy
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image, ImageFilter
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torchvision import transforms

from sklearn.metrics import roc_auc_score, precision_score, f1_score, accuracy_score, confusion_matrix

warnings.filterwarnings("ignore")
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

## 1. Config

The degradation values match `lesion_type_pipeline.ipynb` and are applied at the same 448px size, so a
"60% quality drop" is the same photo for both models. `eval_limit` caps the val/test images per
kind for quick test runs (`None` = all).

In [ ]:
@dataclass
class GateConfig:
    # --- data (paths relative to notebooks/) ---
    ham_csv: str = "data/labels_ham.csv"          # lesion photos: HAM10000 dermoscopy
    pad_csv: str = "data/labels.csv"              # lesion photos: PAD-UFES-20 phone photos (rows with source == "pad_ufes")
    nonskin_dir: str = "data/coco/val2017"        # non-skin photos: COCO val2017 (5,000 everyday photos)
    index_csv: str = "data/gate_index.csv"        # built once by section 2 (rebuilt if missing)
    val_frac: float = 0.10                        # 80/10/10 split, by lesion
    test_frac: float = 0.10

    # --- healthy-skin crops (cut from the corners of lesion photos) ---
    corner_frac: float = 0.30                     # corner square = 30% of the photo's short side
    corner_min_brightness: float = 60.0           # skip dark corners (dermoscope vignette): mean grey level 0-255
    corner_max_dark_frac: float = 0.05            # ... or corners with more than 5% near-black pixels

    # --- image sizes ---
    load_size: int = 448                          # same size the lesion model sees; degradation runs at this size
    img_size: int = 224                           # gate model input

    # --- training ---
    per_epoch: tuple = (("lesion", 6000), ("healthy", 3000), ("nonskin", 3000))  # fresh random sample each epoch, 50/50 yes/no
    batch_size: int = 64
    epochs: int = 15                              # upper limit; early stopping usually ends sooner
    patience: int = 4                             # stop after this many epochs without a new best val AUC
    early_stop_quality_drop: float = 60.0         # model selection: val AUC at this quality drop
    lr: float = 1e-3                              # new yes/no output layer
    backbone_lr: float = 3e-4                     # pretrained MobileNetV3 layers
    weight_decay: float = 1e-4
    lesion_crop_min_scale: float = 0.3            # lesion views: random square crop (30-100%) that keeps the photo's centre
    healthy_crop_min_frac: float = 0.20           # healthy views: corner crop of 20-30% of the short side
    degrade_prob: float = 0.7                     # chance a training image gets degraded ...
    train_degrade_max_pct: float = 80.0           # ... by a random 0-80% quality drop

    # --- degradation model (same values as lesion_type_pipeline.ipynb) ---
    jpeg_quality_floor: int = 15
    blur_radius_max: float = 4.0
    noise_std_max: float = 0.12
    downsample_factor_max: float = 6.0

    # --- threshold + evaluation ---
    target_sensitivity: float = 0.99              # "safety" threshold lets >= 99% of val lesions through ...
    threshold_quality_drop: float = 60.0          # ... both thresholds are set on val photos at this quality drop
    threshold_mode: str = "safety"                # which threshold the Pi uses: "safety" or "accuracy" (decide after results)
    eval_quality_levels: tuple = (0, 20, 40, 60, 80)
    eval_limit: int = None                        # max val/test images per kind (quick runs); None = all

    # --- export ---
    onnx_path: str = "outputs/lesion_gate_mobilenet_v3_small_fp32.onnx"
    onnx_int8_path: str = "outputs/lesion_gate_mobilenet_v3_small_int8.onnx"
    gate_config_path: str = "outputs/lesion_gate_config.json"   # threshold, temperature, sizes: what the Pi needs
    lesion_onnx_path: str = "outputs/skin_lesion_efficientnet_b0_fp32.onnx"  # step 2, from lesion_type_pipeline.ipynb
    lesion_img_size: int = 448
    class_names_path: str = "outputs/class_names.json"


CFG = GateConfig()
os.makedirs("outputs", exist_ok=True)
CFG

## 2. Dataset: lesion vs. healthy skin vs. non-skin

- **lesion** (label 1): every HAM10000 and PAD-UFES-20 photo.
- **healthy** (label 0): a corner square of the same photos, where there is usually only skin.
  Corners that are too dark (the black ring some dermoscopes leave) are skipped. Cutting healthy
  patches out of skin photos is also how other work built healthy-skin sets (see the sources doc).
- **nonskin** (label 0): COCO val2017 everyday photos (rooms, streets, objects, people).

Checking every photo's corners takes a minute or two, so the result is saved to `CFG.index_csv`
and reused. The split is by lesion: a photo and its healthy corner are always in the same split.

In [ ]:
def corner_boxes(w, h, frac):
    """Top-left (x, y) of the 4 corner squares and their side length."""
    c = int(min(w, h) * frac)
    return [(0, 0), (w - c, 0), (0, h - c), (w - c, h - c)], c


def usable_corners(path, cfg):
    """Indices (0-3) of corners that look like plain skin, not the dark dermoscope ring."""
    grey = Image.open(path).convert("L")
    corners, c = corner_boxes(*grey.size, cfg.corner_frac)
    ok = []
    for k, (x, y) in enumerate(corners):
        arr = np.asarray(grey.crop((x, y, x + c, y + c)), dtype=np.float32)
        if arr.mean() >= cfg.corner_min_brightness and (arr < 30).mean() <= cfg.corner_max_dark_frac:
            ok.append(k)
    return ok


def split_by_group(groups, cfg, seed=SEED):
    uniq = np.array(sorted(set(groups)))
    np.random.RandomState(seed).shuffle(uniq)
    n_test, n_val = int(len(uniq) * cfg.test_frac), int(len(uniq) * cfg.val_frac)
    split = {g: "test" for g in uniq[:n_test]}
    split.update({g: "val" for g in uniq[n_test:n_test + n_val]})
    split.update({g: "train" for g in uniq[n_test + n_val:]})
    return groups.map(split)


def build_index(cfg):
    ham = pd.read_csv(cfg.ham_csv)[["image_path", "lesion_id"]].assign(source="ham10000")
    pad = pd.read_csv(cfg.pad_csv)
    pad = pad[pad["source"] == "pad_ufes"][["image_path", "lesion_id"]].assign(source="pad_ufes")
    lesions = pd.concat([ham, pad], ignore_index=True).rename(columns={"lesion_id": "group"})
    print(f"checking corners of {len(lesions)} lesion photos ...")
    lesions["corners"] = [",".join(map(str, usable_corners(p, cfg))) for p in lesions["image_path"]]
    lesions["split"] = split_by_group(lesions["group"], cfg)

    paths = sorted(p.as_posix() for p in Path(cfg.nonskin_dir).glob("*.jpg"))
    nonskin = pd.DataFrame({"image_path": paths, "source": "coco", "corners": ""})
    nonskin["group"] = nonskin["image_path"]
    nonskin["split"] = split_by_group(nonskin["group"], cfg)
    return pd.concat([lesions, nonskin], ignore_index=True)


if Path(CFG.index_csv).exists():
    index_df = pd.read_csv(CFG.index_csv, keep_default_na=False)
else:
    index_df = build_index(CFG)
    index_df.to_csv(CFG.index_csv, index=False)
index_df["corners"] = index_df["corners"].astype(str)


def kind_tables(index_df, split, cfg, fixed_corner):
    """lesion / healthy / nonskin rows of one split. Val/test use one fixed corner per photo."""
    d = index_df[index_df["split"] == split]
    skin = d[d["source"] != "coco"]
    tables = {
        "lesion": skin.assign(kind="lesion", label=1, corner=-1),
        "healthy": skin[skin["corners"] != ""].assign(kind="healthy", label=0, corner=-1),
        "nonskin": d[d["source"] == "coco"].assign(kind="nonskin", label=0, corner=-1),
    }
    if fixed_corner:
        rng = np.random.RandomState(SEED)
        tables["healthy"]["corner"] = [int(rng.choice(c.split(","))) for c in tables["healthy"]["corners"]]
        if cfg.eval_limit:
            tables = {k: t.sample(n=min(cfg.eval_limit, len(t)), random_state=SEED) for k, t in tables.items()}
    return {k: t.reset_index(drop=True) for k, t in tables.items()}


train_sets = kind_tables(index_df, "train", CFG, fixed_corner=False)
val_df = pd.concat(kind_tables(index_df, "val", CFG, fixed_corner=True).values(), ignore_index=True)
test_df = pd.concat(kind_tables(index_df, "test", CFG, fixed_corner=True).values(), ignore_index=True)

print("train:", {k: len(t) for k, t in train_sets.items()})
print("val:  ", val_df["kind"].value_counts().to_dict())
print("test: ", test_df["kind"].value_counts().to_dict())
n_skin = (index_df["source"] != "coco").sum()
print(f"photos with at least one usable healthy corner: {(index_df['corners'] != '').sum() - 0} of {n_skin}")

## 3. Degradation model and photo views

The same one-knob degradation as `lesion_type_pipeline.ipynb` (JPEG, blur, noise, downsampling scaled
together from 0-100%). `load_view` turns a row into the image the model sees:
- **lesion**: in training, a random square crop that keeps the photo's centre (where the lesion
  is), so lesions appear at many zoom levels; in val/test, the whole photo.
- **healthy**: a corner square. Training varies its size; val/test use a fixed corner.
- **nonskin**: the whole photo.

Every view is resized to 448px (where degradation runs), then to 224px for the model. Zooming in on
both lesion and healthy crops stops the model learning "zoomed-in = healthy".

In [ ]:
def _severity(pct, max_val, min_val=0.0):
    pct = max(0.0, min(100.0, pct))
    return min_val + (max_val - min_val) * (pct / 100.0)


def jpeg_compress(img, pct, cfg):
    quality = int(100 - _severity(pct, 100 - cfg.jpeg_quality_floor))
    quality = max(cfg.jpeg_quality_floor, quality)
    buf = io.BytesIO()
    img.save(buf, format="JPEG", quality=quality)
    buf.seek(0)
    return Image.open(buf).convert("RGB")


def gaussian_blur(img, pct, cfg):
    radius = _severity(pct, cfg.blur_radius_max)
    if radius <= 0:
        return img
    return img.filter(ImageFilter.GaussianBlur(radius=radius))


def add_sensor_noise(img, pct, cfg):
    std = _severity(pct, cfg.noise_std_max)
    if std <= 0:
        return img
    arr = np.asarray(img).astype(np.float32) / 255.0
    noisy = np.clip(arr + np.random.normal(0, std, arr.shape), 0, 1) * 255.0
    return Image.fromarray(noisy.astype(np.uint8))


def downsample_upsample(img, pct, cfg):
    factor = 1.0 + _severity(pct, cfg.downsample_factor_max - 1.0)
    if factor <= 1.0:
        return img
    w, h = img.size
    small = img.resize((max(1, int(w / factor)), max(1, int(h / factor))), Image.BILINEAR)
    return small.resize((w, h), Image.BILINEAR)


def degrade_image(img, quality_drop_pct, cfg=None):
    cfg = cfg or CFG
    img = jpeg_compress(img, quality_drop_pct, cfg)
    img = gaussian_blur(img, quality_drop_pct, cfg)
    img = add_sensor_noise(img, quality_drop_pct, cfg)
    img = downsample_upsample(img, quality_drop_pct, cfg)
    return img


def load_view(row, cfg, train=False):
    img = Image.open(row["image_path"]).convert("RGB")
    w, h = img.size
    if row["kind"] == "healthy":
        k = random.choice([int(c) for c in row["corners"].split(",")]) if train else int(row["corner"])
        frac = random.uniform(cfg.healthy_crop_min_frac, cfg.corner_frac) if train else cfg.corner_frac
        corners, c = corner_boxes(w, h, frac)
        # a smaller crop stays anchored to the same image corner
        x = 0 if k in (0, 2) else w - c
        y = 0 if k in (0, 1) else h - c
        img = img.crop((x, y, x + c, y + c))
    elif row["kind"] == "lesion" and train:
        side = int(min(w, h) * random.uniform(cfg.lesion_crop_min_scale, 1.0))
        cx, cy = w // 2, h // 2
        x = random.randint(max(0, cx - side), min(w - side, cx))
        y = random.randint(max(0, cy - side), min(h - side, cy))
        img = img.crop((x, y, x + side, y + side))
    return img.resize((cfg.load_size, cfg.load_size), Image.BILINEAR)


# preview: one photo of each kind, clean and at a 60% quality drop
preview = pd.concat([val_df[val_df["kind"] == k].head(1) for k in ("lesion", "healthy", "nonskin")])
fig, axes = plt.subplots(2, 3, figsize=(9, 6))
for j, (_, row) in enumerate(preview.iterrows()):
    view = load_view(row, CFG)
    axes[0, j].imshow(view); axes[0, j].set_title(f"{row['kind']} (clean)")
    axes[1, j].imshow(degrade_image(view, 60, CFG)); axes[1, j].set_title(f"{row['kind']} (60% drop)")
for ax in axes.ravel():
    ax.axis("off")
plt.tight_layout(); plt.show()

## 4. Model: MobileNetV3-Small

About 1.5M parameters (~6 MB in FP32), small and fast enough for the Pi. Pretrained on ImageNet,
with the last layer replaced by one "is a lesion" output. The whole model is fine-tuned: the
pretrained layers at a smaller learning rate than the new output layer. Inputs use ImageNet
mean/std, like the lesion model.

In [ ]:
IMAGENET_MEAN, IMAGENET_STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

preprocess = transforms.Compose([
    transforms.Resize((CFG.img_size, CFG.img_size)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

model = torchvision.models.mobilenet_v3_small(weights="IMAGENET1K_V1")
model.classifier[-1] = nn.Linear(model.classifier[-1].in_features, 1)  # one logit: "is a lesion"
model = model.to(DEVICE)
print(f"MobileNetV3-Small: {sum(p.numel() for p in model.parameters()):,} parameters")

## 5. Evaluation helpers

Defined before training because early stopping scores the validation set every epoch.
`predict` works for both the PyTorch model and the ONNX files (sections 9-10). Degradation noise
is seeded per quality level, so every run scores the same degraded images.

Metrics, with "lesion" as the positive class:
- **sensitivity**: share of real lesions the gate lets through. The number that matters most.
- **specificity**: share of non-lesion photos it rejects, also split into healthy skin and non-skin.
- **precision**: of the photos it lets through, the share that really show a lesion.
- **F1**, **accuracy**, **AUC** (threshold-free: how well it ranks lesions above non-lesions).

In [ ]:
def predict(run_batch, df, quality_drop_pct, cfg):
    """run_batch: (n, 3, H, W) float tensor -> (n,) numpy outputs."""
    state = np.random.get_state()
    np.random.seed(SEED + int(quality_drop_pct))
    outputs = []
    for start in range(0, len(df), cfg.batch_size):
        batch_df = df.iloc[start:start + cfg.batch_size]
        imgs = [load_view(row, cfg) for _, row in batch_df.iterrows()]
        if quality_drop_pct > 0:
            imgs = [degrade_image(im, quality_drop_pct, cfg) for im in imgs]
        outputs.append(run_batch(torch.stack([preprocess(im) for im in imgs])))
    np.random.set_state(state)
    return np.concatenate(outputs)


@torch.no_grad()
def torch_logits(x):
    model.eval()
    return model(x.to(DEVICE)).squeeze(1).float().cpu().numpy()


def gate_metrics(df, probs, threshold, quality_drop_pct):
    labels = df["label"].values
    preds = (probs >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(labels, preds, labels=[0, 1]).ravel()
    metrics = {
        "quality_drop_pct": quality_drop_pct,
        "accuracy": accuracy_score(labels, preds),
        "sensitivity": tp / max(tp + fn, 1),
        "specificity": tn / max(tn + fp, 1),
        "precision": precision_score(labels, preds, zero_division=0),
        "f1": f1_score(labels, preds, zero_division=0),
        "auc": roc_auc_score(labels, probs),
    }
    for kind in ("healthy", "nonskin"):
        mask = (df["kind"] == kind).values
        metrics[f"specificity_{kind}"] = float((preds[mask] == 0).mean()) if mask.any() else float("nan")
    return metrics


def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))

## 6. Training

Each epoch draws a fresh random sample (`CFG.per_epoch`): 6,000 lesion views and 6,000 non-lesion
views (3,000 healthy + 3,000 non-skin), so the two answers are balanced. Training images get random
flips, 90-degree rotations, small colour shifts, and (70% of the time) a random 0-80% quality drop.

AdamW with two learning rates and a cosine schedule. Early stopping keeps the epoch with the best
validation AUC at a 60% quality drop and restores its weights.

In [ ]:
color_jitter = transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.02)


def train_view(row, cfg):
    img = load_view(row, cfg, train=True)
    if random.random() < 0.5:
        img = img.transpose(Image.Transpose.FLIP_LEFT_RIGHT)
    img = img.rotate(90 * random.randint(0, 3))
    img = color_jitter(img)
    if random.random() < cfg.degrade_prob:
        img = degrade_image(img, random.uniform(0, cfg.train_degrade_max_pct), cfg)
    return img


def sample_epoch(train_sets, cfg, rng):
    parts = [train_sets[kind].sample(n=min(n, len(train_sets[kind])), random_state=rng.randint(1 << 30))
             for kind, n in cfg.per_epoch]
    return pd.concat(parts).sample(frac=1, random_state=rng.randint(1 << 30)).reset_index(drop=True)


def train_gate(model, train_sets, val_df, cfg):
    head_params = list(model.classifier[-1].parameters())
    head_ids = {id(p) for p in head_params}
    body_params = [p for p in model.parameters() if id(p) not in head_ids]
    opt = torch.optim.AdamW([{"params": body_params, "lr": cfg.backbone_lr},
                             {"params": head_params, "lr": cfg.lr}], weight_decay=cfg.weight_decay)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=cfg.epochs)
    rng = np.random.RandomState(SEED)
    best_auc, best_state, best_epoch, bad = -1.0, None, -1, 0

    for epoch in range(cfg.epochs):
        model.train()
        epoch_df = sample_epoch(train_sets, cfg, rng)
        losses = []
        for start in range(0, len(epoch_df), cfg.batch_size):
            batch_df = epoch_df.iloc[start:start + cfg.batch_size]
            x = torch.stack([preprocess(train_view(row, cfg)) for _, row in batch_df.iterrows()]).to(DEVICE)
            y = torch.tensor(batch_df["label"].values, dtype=torch.float32, device=DEVICE)
            loss = F.binary_cross_entropy_with_logits(model(x).squeeze(1), y)
            opt.zero_grad(); loss.backward(); opt.step()
            losses.append(loss.item())
        sched.step()

        val_logits = predict(torch_logits, val_df, cfg.early_stop_quality_drop, cfg)
        val_auc = roc_auc_score(val_df["label"].values, val_logits)
        is_best = val_auc > best_auc
        print(f"epoch {epoch}: loss={np.mean(losses):.4f} val_auc@{cfg.early_stop_quality_drop:.0f}%={val_auc:.4f}"
              + ("  *best*" if is_best else ""))
        if is_best:
            best_auc, best_epoch, bad = val_auc, epoch, 0
            best_state = copy.deepcopy(model.state_dict())
        else:
            bad += 1
            if bad >= cfg.patience:
                print(f"early stop: no improvement for {cfg.patience} epochs")
                break

    model.load_state_dict(best_state)
    print(f"restored best epoch {best_epoch} (val AUC at {cfg.early_stop_quality_drop:.0f}% drop = {best_auc:.4f})")
    return model


model = train_gate(model, train_sets, val_df, CFG)

## 7. Threshold and calibration

**Calibration (temperature scaling):** a raw model output of "0.95" isn't necessarily right 95% of
the time. Temperature scaling (Guo et al., ICML 2017) divides the logit by one number `T`, fitted on
the validation set, so the probability shown to the patient matches how often it's right. It
doesn't change which photos rank higher, so AUC is unchanged. ECE (expected calibration error) is
the average gap between confidence and accuracy, lower is better.

**Thresholds:** the gate lets a photo through when its calibrated probability is at least the
threshold. Two are computed on validation photos at a 60% quality drop, and both are reported:
- **safety**: the highest value that still lets at least 99% (`CFG.target_sensitivity`) of real
  lesions through. It rarely tells someone with a lesion "nothing found", at some cost in accuracy.
- **accuracy**: the value with the highest validation accuracy. It misses more real lesions.

`CFG.threshold_mode` picks the one the Pi uses (`THRESHOLD`). Choose after seeing section 8.

In [ ]:
def expected_calibration_error(labels, probs, n_bins=10):
    bins = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    for lo, hi in zip(bins[:-1], bins[1:]):
        mask = (probs > lo) & (probs <= hi) if lo > 0 else (probs >= lo) & (probs <= hi)
        if mask.any():
            ece += mask.mean() * abs(labels[mask].mean() - probs[mask].mean())
    return ece


val_labels = val_df["label"].values
val_logits = {q: predict(torch_logits, val_df, q, CFG) for q in sorted({0, CFG.threshold_quality_drop})}

# fit the temperature on clean + degraded validation logits together
fit_logits = torch.tensor(np.concatenate(list(val_logits.values())), dtype=torch.float32)
fit_labels = torch.tensor(np.concatenate([val_labels] * len(val_logits)), dtype=torch.float32)
log_t = torch.zeros(1, requires_grad=True)
lbfgs = torch.optim.LBFGS([log_t], lr=0.1, max_iter=200)

def _closure():
    lbfgs.zero_grad()
    loss = F.binary_cross_entropy_with_logits(fit_logits / log_t.exp(), fit_labels)
    loss.backward()
    return loss

lbfgs.step(_closure)
TEMPERATURE = float(log_t.exp())

for q, logits in val_logits.items():
    print(f"val ECE at {q:.0f}% drop: before {expected_calibration_error(val_labels, sigmoid(logits)):.4f} "
          f"-> after {expected_calibration_error(val_labels, sigmoid(logits / TEMPERATURE)):.4f}")

val_probs = sigmoid(val_logits[CFG.threshold_quality_drop] / TEMPERATURE)
# "safety": highest value that still lets target_sensitivity of val lesions through
lesion_probs = np.sort(val_probs[val_labels == 1])
n_allowed_misses = int(np.floor((1 - CFG.target_sensitivity) * len(lesion_probs)))
# "accuracy": the value with the highest val accuracy
candidates = np.unique(val_probs)
val_accuracies = [((val_probs >= t).astype(int) == val_labels).mean() for t in candidates]
THRESHOLDS = {"safety": float(lesion_probs[n_allowed_misses]),
              "accuracy": float(candidates[int(np.argmax(val_accuracies))])}
THRESHOLD = THRESHOLDS[CFG.threshold_mode]
print(f"temperature = {TEMPERATURE:.3f}, thresholds: "
      + ", ".join(f"{name} {t:.4f}" for name, t in THRESHOLDS.items()) + f" (Pi uses {CFG.threshold_mode})")

for q, logits in val_logits.items():
    for name, t in THRESHOLDS.items():
        m = gate_metrics(val_df, sigmoid(logits / TEMPERATURE), t, q)
        print(f"val at {q:.0f}% drop, {name} threshold: accuracy {m['accuracy']:.3f}, "
              f"sensitivity {m['sensitivity']:.3f}, specificity {m['specificity']:.3f} "
              f"(healthy {m['specificity_healthy']:.3f}, non-skin {m['specificity_nonskin']:.3f})")

## 8. Test results vs. quality drop

The same sweep as the lesion model, on the held-out test set, once per threshold from section 7.
Read claims off these tables, for example the gate's accuracy and sensitivity at a 60% quality
drop. `sensitivity` is the share of real lesions let through; `specificity` the share of non-lesion
photos rejected.

In [ ]:
SUMMARY_COLS = ["quality_drop_pct", "accuracy", "sensitivity", "specificity", "specificity_healthy",
                "specificity_nonskin", "precision", "f1", "auc"]

test_probs = {q: sigmoid(predict(torch_logits, test_df, q, CFG) / TEMPERATURE) for q in CFG.eval_quality_levels}
gate_results = {
    name: pd.DataFrame([gate_metrics(test_df, test_probs[q], t, q) for q in CFG.eval_quality_levels])
    for name, t in THRESHOLDS.items()
}
for name, results_df in gate_results.items():
    print(f"\n===== {name} threshold ({THRESHOLDS[name]:.4f}) =====")
    print(results_df[SUMMARY_COLS].round(4))
gate_results_df = gate_results[CFG.threshold_mode]

fig, ax = plt.subplots(figsize=(7, 4.5))
for name, style in (("safety", "-"), ("accuracy", "--")):
    results_df = gate_results[name]
    for col, label, color in [("accuracy", "accuracy", "C0"),
                              ("sensitivity", "sensitivity (lesions let through)", "C1"),
                              ("specificity_healthy", "specificity: healthy skin", "C2"),
                              ("specificity_nonskin", "specificity: non-skin", "C3")]:
        ax.plot(results_df["quality_drop_pct"], results_df[col], style, marker="o", color=color,
                label=f"{label} ({name})")
ax.set_xlabel("simulated quality drop (%)"); ax.set_ylabel("rate"); ax.set_ylim(0, 1.02)
ax.set_title("Lesion gate on the test set (solid = safety, dashed = accuracy threshold)")
ax.legend(fontsize=7, ncol=2); ax.grid(alpha=0.3)
plt.tight_layout(); plt.savefig("outputs/lesion_gate_quality_curve.png", dpi=150); plt.show()

## 9. Export: ONNX FP32 and INT8

The exported model includes the temperature and the sigmoid, so the Pi gets a calibrated
probability (`p_lesion`) directly and compares it with the threshold saved in
`CFG.gate_config_path`. The Pi preprocessing: resize the photo to 448x448, then to 224x224, then
normalize with ImageNet mean/std.

INT8 is exported too, but MobileNetV3 has the same kinds of layers that broke INT8 for
EfficientNet-B0 (hard-swish, squeeze-excite, depthwise convolutions), so section 10 checks it
before it's trusted.

In [ ]:
import onnx
import onnxruntime as ort
from onnxruntime.quantization import quantize_dynamic, QuantType


class DeployableGate(nn.Module):
    """MobileNetV3 + temperature + sigmoid: outputs the calibrated probability of a lesion."""

    def __init__(self, model, temperature):
        super().__init__()
        self.model = copy.deepcopy(model).cpu().eval()
        self.register_buffer("temperature", torch.tensor(float(temperature)))

    def forward(self, x):
        return torch.sigmoid(self.model(x) / self.temperature)  # (batch, 1)


deployable = DeployableGate(model, TEMPERATURE).eval()
torch.onnx.export(
    deployable, torch.randn(1, 3, CFG.img_size, CFG.img_size), CFG.onnx_path,
    input_names=["pixel_values"], output_names=["p_lesion"],
    dynamic_axes={"pixel_values": {0: "batch"}, "p_lesion": {0: "batch"}},
    opset_version=18,
    dynamo=False,  # the dynamo exporter breaks on dynamic_axes
)
onnx.checker.check_model(onnx.load(CFG.onnx_path))
quantize_dynamic(model_input=CFG.onnx_path, model_output=CFG.onnx_int8_path, weight_type=QuantType.QInt8)

gate_config = {
    "threshold": THRESHOLD, "threshold_mode": CFG.threshold_mode, "thresholds": THRESHOLDS,
    "temperature": TEMPERATURE, "target_sensitivity": CFG.target_sensitivity,
    "load_size": CFG.load_size, "img_size": CFG.img_size, "mean": IMAGENET_MEAN, "std": IMAGENET_STD,
    "input": "pixel_values", "output": "p_lesion",
}
with open(CFG.gate_config_path, "w") as f:
    json.dump(gate_config, f, indent=2)

for path in (CFG.onnx_path, CFG.onnx_int8_path):
    print(f"{path}: {os.path.getsize(path) / 1e6:.1f} MB")
print(f"saved threshold + preprocessing to {CFG.gate_config_path}")

## 10. Re-measure on the exported ONNX files

The same test sweep through the FP32 and INT8 ONNX files (CPU). FP32 should match section 8. If
INT8 falls far below FP32, ship the FP32 file: it's only a few MB.

In [ ]:
onnx_gate_results = {}
for name, path in [("FP32", CFG.onnx_path), ("INT8", CFG.onnx_int8_path)]:
    session = ort.InferenceSession(path, providers=["CPUExecutionProvider"])
    run_onnx = lambda x, s=session: s.run(None, {"pixel_values": x.numpy()})[0][:, 0]
    probs = {q: predict(run_onnx, test_df, q, CFG) for q in CFG.eval_quality_levels}
    for mode, t in THRESHOLDS.items():
        results_df = pd.DataFrame([gate_metrics(test_df, probs[q], t, q) for q in CFG.eval_quality_levels])
        onnx_gate_results[(name, mode)] = results_df
        print(f"\n===== {name} ONNX, {mode} threshold ({t:.4f}) =====")
        print(results_df[SUMMARY_COLS].round(4))

## 11. The full two-step pipeline on example photos

`triage(photo)` is what the Pi does for one photo, using only the exported files:
1. Gate: `p_lesion` below the threshold means "no skin lesion found", and the lesion model doesn't run.
2. Otherwise the lesion-type model (`CFG.lesion_onnx_path`, from `lesion_type_pipeline.ipynb`) gives
   its top 3 types, averaged with the mirror image like its TTA results.

The gate's confidence is calibrated; the lesion model's probabilities are not yet.

In [ ]:
gate_session = ort.InferenceSession(CFG.onnx_path, providers=["CPUExecutionProvider"])
lesion_session = ort.InferenceSession(CFG.lesion_onnx_path, providers=["CPUExecutionProvider"])
with open(CFG.class_names_path) as f:
    LESION_CLASSES = json.load(f)
with open(CFG.gate_config_path) as f:
    GATE = json.load(f)

lesion_preprocess = transforms.Compose([
    transforms.Resize((CFG.lesion_img_size, CFG.lesion_img_size)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])


def triage(photo):
    photo = photo.convert("RGB")
    x = preprocess(photo.resize((GATE["load_size"], GATE["load_size"]), Image.BILINEAR)).unsqueeze(0).numpy()
    p_lesion = float(gate_session.run(None, {"pixel_values": x})[0][0, 0])
    if p_lesion < GATE["threshold"]:
        return {"lesion_found": False, "p_lesion": p_lesion}
    pair = [photo, photo.transpose(Image.Transpose.FLIP_LEFT_RIGHT)]
    logits = lesion_session.run(None, {"pixel_values": torch.stack([lesion_preprocess(im) for im in pair]).numpy()})[0]
    probs = np.exp(logits - logits.max(axis=1, keepdims=True))
    probs = (probs / probs.sum(axis=1, keepdims=True)).mean(axis=0)
    top3 = np.argsort(probs)[::-1][:3]
    return {"lesion_found": True, "p_lesion": p_lesion,
            "top3": [(LESION_CLASSES[i], float(probs[i])) for i in top3]}


examples = [(row, q) for kind in ("lesion", "healthy", "nonskin")
            for row in [test_df[test_df["kind"] == kind].iloc[0]] for q in (0, 60)]
fig, axes = plt.subplots(1, len(examples), figsize=(3 * len(examples), 3.6))
np.random.seed(SEED)
for ax, (row, q) in zip(axes, examples):
    photo = load_view(row, CFG)
    if q > 0:
        photo = degrade_image(photo, q, CFG)
    result = triage(photo)
    if result["lesion_found"]:
        name, p = result["top3"][0]
        verdict = f"lesion ({result['p_lesion']:.0%})\ntop type: {name} {p:.0%}"
    else:
        verdict = f"no lesion found\n(p_lesion {result['p_lesion']:.0%})"
    ax.imshow(photo); ax.axis("off")
    ax.set_title(f"{row['kind']}, {q}% drop\n{verdict}", fontsize=9)
plt.tight_layout(); plt.show()

## 12. Limitations and next steps

- **No real Pi photos yet.** The healthy skin comes from the corners of close-up lesion photos (many
  are dermoscopy), and non-skin from COCO. Test the gate on photos taken with the actual Pi camera
  before trusting its numbers.
- **PAD photos often have a pen circle around the lesion**, which the gate could learn as a shortcut.
  Pi photos won't have one.
- **Corners aren't always clean**: a large lesion can reach a corner, and some PAD corners show
  background. This makes the healthy class a little noisy.
- **Calibration is fitted on this validation mix** (about 50% lesions). In real use the mix differs,
  so treat the shown confidence as approximate.
- Possible additions: a dedicated healthy-skin dataset (e.g. iToBoS non-lesion crops), and a
  "photo too poor to judge" check.